In [1]:
from airflow.sdk import dag, task
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param
from airflow.sdk.definitions.param import ParamsDict
from sqlalchemy import create_engine, VARCHAR, Integer,Date, inspect
import os
import pandas as pd
from datetime import datetime
import zipfile
import numpy as np
import requests
from datetime import datetime
from pathlib import Path
import shutil
import boto3
from botocore.config import Config


c:\Users\Nelio\anaconda3\Lib\site-packages\airflow\__init__.py:45: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(
2026-04-15T14:39:48.336308Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T14:39:48.344273Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T14:39:48.351591Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T14:39:48.359553Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-04-1

In [35]:
def candidato(file_csv):
    colunas_leitura = ['dt_geracao', 'hh_geracao', 'ano_eleicao', 'cd_tipo_eleicao',
       'nm_tipo_eleicao', 'nr_turno', 'cd_eleicao', 'ds_eleicao', 'dt_eleicao',
       'tp_abrangencia', 'sg_uf', 'sg_ue', 'nm_ue', 'cd_cargo', 'ds_cargo',
       'sq_candidato', 'nr_candidato', 'nm_candidato', 'nm_urna_candidato',
       'nm_social_candidato', 'nr_cpf_candidato', 'ds_email',
       'cd_situacao_candidatura', 'ds_situacao_candidatura', 'tp_agremiacao',
       'nr_partido', 'sg_partido', 'nm_partido', 'nr_federacao',
       'nm_federacao', 'sg_federacao', 'ds_composicao_federacao',
       'sq_coligacao', 'nm_coligacao', 'ds_composicao_coligacao',
       'sg_uf_nascimento', 'dt_nascimento', 'nr_titulo_eleitoral_candidato',
       'cd_genero', 'ds_genero', 'cd_grau_instrucao', 'ds_grau_instrucao',
       'cd_estado_civil', 'ds_estado_civil', 'cd_cor_raca', 'ds_cor_raca',
       'cd_ocupacao', 'ds_ocupacao', 'cd_sit_tot_turno', 'ds_sit_tot_turno',
       'id_candidato', 'nome_foto', 'url_foto']
    dfs = []
    lista_links = []
    # Configurações do Cloudflare
    account_id = '8e2ebba9301e868cfdedc13086d6c140'
    access_key = 'a35b4c037c701ade40b95d0c48e4b4b3'
    secret_key = '9eb9f6eac22bb9a13a90f36ab18092e455fef1ead93ec303fe94c0bf2ae90e42'
    bucket_name = 'tre'
    folder_local = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\jpg' ##os.getenv('CANDIDATO_FOTOS_DIR', '/opt/airflow/TRE2026/archives/RR/candidato/jpg')
    folder_remote = 'fotosrr/'               

    s3 = boto3.client(
    service_name='s3',
    endpoint_url=f'https://{account_id}.r2.cloudflarestorage.com',
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='auto', 
    config=Config(signature_version='s3v4')
    )
    base_url = "https://pub-e9ede5cdf96443d58340a9a3b62a2816.r2.dev/fotosrr/" 
    # engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")

    for i in os.listdir(file_csv):
        full_path = os.path.join(file_csv, i)
        if full_path.endswith('RR.csv'):
            print(f"Lendo arquivo: {full_path}")
            df = pd.read_csv(full_path, encoding='latin-1', sep=';',usecols=lambda col: col.strip().lower() in colunas_leitura,dtype=str)
            df.columns = [col.strip().lower() for col in df.columns]
            df['sg_ue'] = df['sg_ue'].astype(str).str.lstrip("0")
            df.loc[df['nm_ue'] == "RORAIMA", 'sg_ue'] = '3018'
            df['nm_candidato'] = (
                df['nm_urna_candidato'] 
                .fillna('')
                .astype(str)
                .str.normalize('NFKD')
                .str.encode('ascii', 'ignore')
                .str.decode('utf-8')
                .str.replace(r'[^A-Za-z0-9]+', '', regex=True)
            )
            df['id_candidato'] = df['ano_eleicao'] + df['nr_turno'] +  df['cd_cargo'] + df['sg_ue'] +  df['nr_candidato'] + df['nm_candidato']
                  
            dfs.append(df)


    dfconsulta = pd.concat(dfs, ignore_index=True)

    for file in os.listdir(folder_local):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            lista_links.append({
                "Nome_Foto": file,
                "URL_Foto": base_url + file
            })
    fotos = pd.DataFrame(lista_links)
    fotos['idCandidatos'] = fotos['Nome_Foto'].str[2:14]

    df = pd.merge(
    dfconsulta, 
    fotos, 
    how='left', 
    right_on='idCandidatos',
    left_on='sq_candidato').drop('idCandidatos', axis=1) 
    df.drop_duplicates(['id_candidato'], inplace=True)

    # df.to_sql("dcandidato", engine, index=False, if_exists='append')





    return df

In [36]:
df = candidato(file_csv = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv')


Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2020_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2022_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2024_RR.csv


In [38]:
df

,dt_geracao,hh_geracao,ano_eleicao,cd_tipo_eleicao,nm_tipo_eleicao,nr_turno,cd_eleicao,ds_eleicao,dt_eleicao,tp_abrangencia,...,ds_estado_civil,cd_cor_raca,ds_cor_raca,cd_ocupacao,ds_ocupacao,cd_sit_tot_turno,ds_sit_tot_turno,id_candidato,Nome_Foto,URL_Foto
0,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,DIVORCIADO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622333DANIELEDECAMPOSNOVOS,NaN,NaN
1,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622222CASSIODOTAXI,NaN,NaN
2,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),05,INDÍGENA,601,AGRICULTOR,5,SUPLENTE,2020113302611611MARA,NaN,NaN
3,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302610555TANDYLIMA,NaN,NaN
4,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,CASADO(A),03,PARDA,266,PROFESSOR DE ENSINO MÉDIO,5,SUPLENTE,2020113302610456PROFESSORAGNALDO,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3845,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,999,OUTROS,5,SUPLENTE,2024113305010000RONYPETERSON,NaN,NaN
3846,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,2,ELEITO POR QP,2024113305010777SANDRIELYCUNHA,NaN,NaN
3847,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,931,"ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS",5,SUPLENTE,2024113305020789MOACYPEDEMINGAL,NaN,NaN
3848,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,CASADO(A),03,PARDA,257,EMPRESÁRIO,5,SUPLENTE,2024113305020333CHEILADOAUAU,NaN,NaN


In [ ]:
from airflow.sdk import dag, task
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param
from airflow.sdk.definitions.param import ParamsDict
from sqlalchemy import create_engine, VARCHAR, Integer,Date, inspect
import os
import pandas as pd
from datetime import datetime
import zipfile
import numpy as np
import requests
from cosmos import DbtDag, ProjectConfig, ProfileConfig, RenderConfig
from cosmos.profiles import PostgresUserPasswordProfileMapping
from cosmos import DbtTaskGroup, ProjectConfig, ProfileConfig, RenderConfig
from cosmos.profiles import PostgresUserPasswordProfileMapping
from datetime import datetime
from pathlib import Path
import shutil
import boto3
from botocore.config import Config



# Capturando as variáveis do seu ambiente Docker
USER = os.getenv('DB_USER')
PASSWORD = os.getenv('DB_PASSWORD')

params = {
        "Usuario": Param(
        default="José Nélio"
        ,type='string'
        ,enum=["José Nélio"]
        ,description="""
        Escolher usuario 
        """
        )

    ,'database': Param(
        default='eleicao_RR'
        ,type="string"
        ,description="Escolher o nome do banco de archives para subir os archives tratados")

    }

# Configuração de onde o dbt 

DBT_PROJECT_PATH = Path("/opt/airflow/dbt/eleicao")

profile_config = ProfileConfig(
    profile_name="eleicao",
    target_name="dev",
    profile_mapping=PostgresUserPasswordProfileMapping(
        conn_id="postgres_default", # O ID da conexão 
        profile_args={"schema": "public"},
    ),
)


default_args = {
    "owner": "Nelio Cruel",
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

@dag(
    dag_id="dag_eleicao",
    start_date=datetime(2024, 1, 1),
    schedule="@once", 
    params=params,
    catchup=False,
    tags=["cinema"],
    max_active_tasks= 30,
    default_args=default_args
)


def eleicao():
    
    @task
    def unzip_file(zip_path, file_save):
        os.makedirs(file_save, exist_ok=True)

        for file_name in os.listdir(zip_path):
            if file_name.lower().endswith('.zip'):
                zip_file = os.path.join(zip_path, file_name)
                with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                    for member in zip_ref.namelist():
                        nome_arquivo = os.path.basename(member)
                        destino_final = os.path.join(file_save, nome_arquivo)
                        with zip_ref.open(member) as origem, open(destino_final, 'wb') as destino:
                            shutil.copyfileobj(origem, destino)
                        print(f"Extraído: {nome_arquivo}")



    @task
    def local_votacao (path):
        engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")
        colunas_leitura = [
        'dt_geracao', 'hh_geracao', 'aa_eleicao', 'dt_eleicao', 'ds_eleicao',
        'nr_turno', 'sg_uf', 'cd_municipio', 'nm_municipio', 'nr_zona',
        'nr_secao', 'cd_tipo_secao_agregada', 'ds_tipo_secao_agregada',
        'nr_secao_principal', 'nr_local_votacao', 'nm_local_votacao',
        'cd_tipo_local', 'ds_tipo_local', 'ds_endereco', 'nm_bairro', 'nr_cep',
        'nr_telefone_local', 'nr_latitude', 'nr_longitude',
        'cd_situ_local_votacao', 'ds_situ_local_votacao', 'cd_situ_zona',
        'ds_situ_zona', 'cd_situ_secao', 'ds_situ_secao', 'cd_situ_localidade',
        'ds_situ_localidade', 'cd_situ_secao_acessibilidade',
        'ds_situ_secao_acessibilidade', 'qt_eleitor_secao',
        'qt_eleitor_eleicao_federal', 'qt_eleitor_eleicao_estadual',
        'qt_eleitor_eleicao_municipal', 'nr_local_votacao_original',
        'nm_local_votacao_original', 'ds_endereco_locvt_original',
        'id_zona_secao_municipio'
        ]
        # dfs = []
        colunas_num = ['qt_eleitor_secao', 'qt_eleitor_eleicao_federal', 'qt_eleitor_eleicao_estadual','qt_eleitor_eleicao_municipal']
        colunas_data = ['dt_geracao','dt_eleicao']
        colunas_hora = ['hh_geracao']
        for arquivo in os.listdir(path):
            full_path = os.path.join(path, arquivo)
            if full_path.endswith('.csv'):
                print(f"Lendo o arquivo: {full_path}")
                df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str,usecols=lambda col: col.strip().lower() in colunas_leitura, nrows=10)
                df.columns = [col.strip().lower() for col in df.columns]
                for col in colunas_num:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col]).astype('Int64')
                for col in colunas_data:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d/%m/%Y', errors='coerce').dt.date
                for col in colunas_hora:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time
                df['aa_eleicao'] = df['aa_eleicao'].astype(str).str.lstrip("0")
                df['id_zona_secao_municipio'] = df['aa_eleicao'] +  df['nr_turno'] + df['nr_zona'] +  df['nr_secao'] +  df['cd_municipio']
                # dfs.append(df)
                print(f"Carregando o arquivo na base de dados: {arquivo}")
                df.to_sql("local_votacao", engine, index=False, if_exists='append')
                print(f"Arquivo Carregado na base de dados: {arquivo}")


        return print("Todos os arquivos foram processados e carregados na base de dados com sucesso!")

    @task
    def perfil (file_csv):
        engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")
        colunas_leitura = [
        'dt_geracao', 'hh_geracao', 'ano_eleicao', 'sg_uf', 'cd_municipio',
        'nm_municipio', 'nr_zona', 'nr_secao', 'nr_local_votacao',
        'nm_local_votacao', 'cd_genero', 'ds_genero', 'cd_estado_civil',
        'ds_estado_civil', 'cd_faixa_etaria', 'ds_faixa_etaria',
        'cd_grau_escolaridade', 'ds_grau_escolaridade', 'cd_raca_cor',
        'ds_raca_cor', 'cd_identidade_genero', 'ds_identidade_genero',
        'cd_quilombola', 'ds_quilombola', 'cd_interprete_libras',
        'ds_interprete_libras', 'tp_obrigatoriedade_voto',
        'qt_eleitores_perfil', 'qt_eleitores_biometria',
        'qt_eleitores_deficiencia', 'qt_eleitores_inc_nm_social',
        'id_zona_secao_municipio'
        ]
        # dfs = []
        colunas_num = ['qt_eleitores_inc_nm_social', 'qt_eleitores_deficiencia', 'qt_eleitores_biometria', 'qt_eleitores_perfil']
        colunas_data = ['dt_geracao']
        colunas_hora = ['hh_geracao']
        for arquivo in os.listdir(file_csv):
            full_path = os.path.join(file_csv, arquivo)
            if full_path.endswith('.csv'):
                print(f"Lendo arquivo: {full_path}")
                df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str,usecols=lambda col: col.strip().lower() in colunas_leitura,nrows=10)
                df.columns = [col.strip().lower() for col in df.columns]
                for col in colunas_num:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col]).astype('Int64')
                for col in colunas_data:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d/%m/%Y', errors='coerce').dt.date
                for col in colunas_hora:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time
                df['id_zona_secao_municipio'] = df['ano_eleicao'] + df['nr_zona'] + df['nr_secao'] + df['cd_municipio']
                # dfs.append(df)
                print(f"Carregando o arquivo na base de dados: {arquivo}")
                df.to_sql("perfil_eleitorado", engine, index=False, if_exists='append')
                print(f"Arquivo Carregado na base de dados: {arquivo}")

        return print("Todos os arquivos foram processados e carregados na base de dados com sucesso!")
    

    @task
    def turnos (file_csv):
        engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")
        # dfs = []
        colunas_leitura =[
            'dt_geracao', 'hh_geracao', 'ano_eleicao', 'cd_tipo_eleicao',
        'nm_tipo_eleicao', 'cd_pleito', 'dt_pleito', 'nr_turno', 'cd_eleicao',
        'ds_eleicao', 'sg_uf', 'cd_municipio', 'nm_municipio', 'nr_zona',
        'nr_secao', 'nr_local_votacao', 'cd_cargo_pergunta',
        'ds_cargo_pergunta', 'nr_partido', 'sg_partido', 'nm_partido',
        'dt_bu_recebido', 'qt_aptos', 'qt_comparecimento', 'qt_abstencoes',
        'cd_tipo_urna', 'ds_tipo_urna', 'cd_tipo_votavel', 'ds_tipo_votavel',
        'nr_votavel', 'nm_votavel', 'qt_votos', 'nr_urna_efetivada',
        'cd_carga_1_urna_efetivada', 'cd_carga_2_urna_efetivada',
        'cd_flashcard_urna_efetivada', 'dt_carga_urna_efetivada',
        'ds_cargo_pergunta_secao', 'ds_agregadas', 'dt_abertura',
        'dt_encerramento', 'qt_eleitores_biometria_nh', 'dt_emissao_bu',
        'nr_junta_apuradora', 'nr_turma_apuradora', 'id_zona_secao_municipio',
        'nm_candidato', 'id_candidato'
        ]
        colunas_num = ['nr_turma_apuradora', 'nr_junta_apuradora', 'qt_votos', 'qt_aptos', 'qt_comparecimento', 'qt_abstencoes']
        colunas_data = ['dt_geracao', 'dt_pleito']
        colunas_dataHora = ['dt_bu_recebido', 'dt_carga_urna_efetivada', 'dt_abertura', 'dt_encerramento', 'dt_emissao_bu']
        colunas_hora = ['hr_geracao']
        for arquivo in os.listdir(file_csv):
            full_path = os.path.join(file_csv, arquivo)
            if full_path.endswith('.csv'):
                df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str,usecols=lambda col: col.strip().lower() in colunas_leitura,nrows=10)
                for col in colunas_num:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
                for col in colunas_data:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d/%m/%Y', errors='coerce').dt.date
                for col in colunas_hora:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time
                for col in colunas_dataHora:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], dayfirst=True,  format='%d/%m/%Y %H:%M:%S', errors='coerce').dt.date
                df['ID_ZONA_SECAO_MUNICIPIO'] = df['ANO_ELEICAO'] + df['NR_TURNO'] + df['NR_ZONA'] +  df['NR_SECAO'] +  df['CD_MUNICIPIO']
                df['NM_CANDIDATO'] = (
                    df['NM_VOTAVEL']
                    .fillna('')
                    .astype(str)
                    .str.normalize('NFKD')
                    .str.encode('ascii', 'ignore')
                    .str.decode('utf-8')
                    .str.replace(r'[^A-Za-z0-9]+', '', regex=True)
                )
                df['ID_CANDIDATO'] = df['ANO_ELEICAO'] + df['NR_TURNO'] +  df['CD_CARGO_PERGUNTA'] + df['CD_MUNICIPIO'] +  df['NR_VOTAVEL'] + df['NM_CANDIDATO']
                # dfs.append(df)
                print(f"Carregando o arquivo no banco de archives: {arquivo}")
                df.to_sql("resultado", engine, index=False, if_exists='append')
                print(f"Arquivo Carregado no Bando de archives: {arquivo}")

        return print("Todos os arquivos foram processados e carregados no banco de archives com sucesso!")
    
    
    @task
    def unzip_candidato(zip_path, file_save):
        os.makedirs(file_save, exist_ok=True)

        for file_name in os.listdir(zip_path):
            if file_name.lower().endswith('.zip'):
                zip_file = os.path.join(zip_path, file_name)
                with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                    for member in zip_ref.namelist():
                        if member.endswith('/') or '_RR.csv' not in os.path.basename(member):
                            continue
                        nome_arquivo = os.path.basename(member)
                        destino_final = os.path.join(file_save, nome_arquivo)
                        with zip_ref.open(member) as origem, open(destino_final, 'wb') as destino:
                            shutil.copyfileobj(origem, destino)
                        print(f"Extraído: {nome_arquivo}")


    @task
    def candidato(file_csv):

        dfs = []
        lista_links = []
        # Configurações do Cloudflare
        account_id = '8e2ebba9301e868cfdedc13086d6c140'
        access_key = 'a35b4c037c701ade40b95d0c48e4b4b3'
        secret_key = '9eb9f6eac22bb9a13a90f36ab18092e455fef1ead93ec303fe94c0bf2ae90e42'
        bucket_name = 'tre'
        folder_local = os.getenv('CANDIDATO_FOTOS_DIR', '/opt/airflow/TRE2026/archives/RR/candidato/jpg')
        folder_remote = 'fotosrr/'               

        s3 = boto3.client(
        service_name='s3',
        endpoint_url=f'https://{account_id}.r2.cloudflarestorage.com',
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name='auto', 
        config=Config(signature_version='s3v4')
        )
        base_url = "https://pub-e9ede5cdf96443d58340a9a3b62a2816.r2.dev/fotosrr/" 
        engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")

        for i in os.listdir(file_csv):
            full_path = os.path.join(file_csv, i)
            if full_path.endswith('RR.csv'):
                print(f"Lendo arquivo: {full_path}")
                df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str)
                df['SG_UE'] = df['SG_UE'].astype(str).str.lstrip("0")
                df.loc[df['NM_UE'] == "RORAIMA", 'SG_UE'] = '3018'
                df['NM_CANDIDATO'] = (
                    df['NM_URNA_CANDIDATO']
                    .fillna('')
                    .astype(str)
                    .str.normalize('NFKD')
                    .str.encode('ascii', 'ignore')
                    .str.decode('utf-8')
                    .str.replace(r'[^A-Za-z0-9]+', '', regex=True)
                )
                df['ID_CANDIDATO'] = df['ANO_ELEICAO'] + df['NR_TURNO'] +  df['CD_CARGO'] + df['SG_UE'] +  df['NR_CANDIDATO'] + df['NM_CANDIDATO']
                dfs.append(df)


        dfconsulta = pd.concat(dfs, ignore_index=True)

        for file in os.listdir(folder_local):
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                lista_links.append({
                    "Nome_Foto": file,
                    "URL_Foto": base_url + file
                })
        fotos = pd.DataFrame(lista_links)
        fotos['idCandidatos'] = fotos['Nome_Foto'].str[2:14]

        df = pd.merge(
        dfconsulta, 
        fotos, 
        how='left', 
        right_on='idCandidatos',
        left_on='SQ_CANDIDATO').drop('idCandidatos', axis=1)
        df.columns = [col.lower() for col in df.columns]
        df.drop_duplicates(['id_candidato'], inplace=True)

        df.to_sql("dcandidato", engine, index=False, if_exists='append')





        return print("Todos os arquivos foram processados e carregados no banco de archives com sucesso!")







### FIM DA FUNÇÃO / CHAMADAS DAS FUNÇÕES PARA CADA TAREFA ###

    unzip_local_votacao_RR = unzip_file.override(task_id="unzip_local_votacao_RR")(
        zip_path = "/opt/airflow/TRE2026/archives/RR/local_votacao/zip",
        file_save="/opt/airflow/TRE2026/archives/RR/local_votacao/csv"
    )
    etl_local_votacao_RR = local_votacao.override(task_id="etl_local_votacao_RR")(path = '/opt/airflow/TRE2026/archives/RR/local_votacao/csv')
    mover_local_votacao_RR = BashOperator(
    task_id='mover_local_votacao_RR',
    bash_command=f'mv /opt/airflow/TRE2026/archives/RR/local_votacao/zip/*.zip /opt/airflow/TRE2026/archives/RR/local_votacao/processados/'
    )


###-------------------------------------------------------------------------------------------------------------------------------------------------#####

    unzip_perfil_eleitor_RR = unzip_file.override(task_id="unzip_perfil_eleitor_RR")(
    zip_path = "/opt/airflow/TRE2026/archives/RR/perfil_eleitorado_secao/zip",
    file_save="/opt/airflow/TRE2026/archives/RR/perfil_eleitorado_secao/csv",
    )
    perfil_eleitorado_RR = perfil.override(task_id="etl_perfil_eleitorado_RR")(file_csv = '/opt/airflow/TRE2026/archives/RR/perfil_eleitorado_secao/csv')
    mover_perfil_eleitorado_RR = BashOperator(
    task_id='mover_perfil_eleitorado_RR',
    bash_command=f'mv /opt/airflow/TRE2026/archives/RR/perfil_eleitorado_secao/zip/*.zip /opt/airflow/TRE2026/archives/RR/perfil_eleitorado_secao/processados/'
    )

# ###-------------------------------------------------------------------------------------------------------------------------------------------------#####
    unzip_resultado_RR = unzip_file.override(task_id="unzip_resultado_RR")(
        zip_path = "/opt/airflow/TRE2026/archives/RR/turnos/zip",
        file_save="/opt/airflow/TRE2026/archives/RR/turnos/csv",
    )
    resultado_RR = turnos.override(task_id="etl_resultado_RR")(file_csv ='/opt/airflow/TRE2026/archives/RR/turnos/csv')
    mover_resultado_RR = BashOperator(
    task_id='mover_resultado_RR',
    bash_command=f'mv /opt/airflow/TRE2026/archives/RR/turnos/zip/*.zip /opt/airflow/TRE2026/archives/RR/turnos/processados/'
    )

###-------------------------------------------------------------------------------------------------------------------------------------------------#####

    unzip_csv_candidato_RR = unzip_candidato.override(task_id="unzip_csv_candidatos_RR")(
        zip_path = "/opt/airflow/TRE2026/archives/RR/candidato/zip-csv",
        file_save="/opt/airflow/TRE2026/archives/RR/candidato/csv"
    )
    unzip_foto_candidato_RR = unzip_file.override(task_id="unzip_foto_candidatos_RR")(
    zip_path = "/opt/airflow/TRE2026/archives/RR/candidato/zip-fotos",
    file_save="/opt/airflow/TRE2026/archives/RR/candidato/jpg"
)
    etl_candidato_RR = candidato.override(task_id="etl_candidato_RR")(file_csv = '/opt/airflow/TRE2026/archives/RR/candidato/csv')

    mover_candidato_csv_RR = BashOperator(
    task_id='mover_candidato_zip_RR',
    bash_command=f'mv /opt/airflow/TRE2026/archives/RR/candidato/zip-csv/*.zip /opt/airflow/TRE2026/archives/RR/candidato/processados-csv/'
    )
    mover_candidato_jpg_RR = BashOperator(
    task_id='mover_candidato_jpg_RR',
    bash_command=f'mv /opt/airflow/TRE2026/archives/RR/candidato/zip-fotos/*.zip /opt/airflow/TRE2026/archives/RR/candidato/processados-jpg/'
    )

#     dbt_eleicao= DbtTaskGroup(
#             group_id="camada_transformacao_dbt",
#             project_config=ProjectConfig(DBT_PROJECT_PATH),
#             profile_config=profile_config,
#             render_config=RenderConfig(
#                 select=["path:models/cinema"]
#             ),
#             operator_args={"install_deps": True},
#         )


    
    start = EmptyOperator(task_id = 'start')
    end = EmptyOperator(task_id = 'end')


    start >> unzip_local_votacao_RR  >> etl_local_votacao_RR >> mover_local_votacao_RR >> end
    start >> unzip_perfil_eleitor_RR >> perfil_eleitorado_RR >> mover_perfil_eleitorado_RR >> end
    start >> unzip_resultado_RR >> resultado_RR >> mover_resultado_RR >> end
    start >> [unzip_csv_candidato_RR , unzip_foto_candidato_RR ] >> etl_candidato_RR >> [mover_candidato_csv_RR , mover_candidato_jpg_RR] >> end

eleicao()